In [ ]:
# 1.定义技能skill(可以使用md文件)

In [ ]:
#定义技能（TypedDict）

from typing import TypedDict

class Skill(TypedDict):
    """一种可以逐步地，向代理披露的技能"""
    name        : str       # 技能的唯一标识
    description : str       # 在系统提示词中，使用1～2句简短的提示词表示
    content     : str       # 包含详细完整的技能内容

In [2]:
SKILLS: list[Skill] = [
    {
        "name"          :"sales_analytics",
        "description"   :"用于销售数据分析的数据库模式和业务逻辑，包括客户、订单和收入",
        "content"       :"""# 销售分析模式

## 数据表

### customers (客户表)
- customer_id (主键)
- name (姓名)
- email (邮箱)
- signup_date (注册日期)
- status (状态: active活跃/inactive非活跃)
- customer_tier (客户等级: bronze铜/silver银/gold金/platinum铂金)

### orders (订单表)
- order_id (主键)
- customer_id (外键 -> customers)
- order_date (订单日期)
- status (状态: pending待处理/completed已完成/cancelled已取消/refunded已退款)
- total_amount (总金额)
- sales_region (销售区域: north北/south南/east东/west西)

### order_items (订单明细表)
- item_id (主键)
- order_id (外键 -> orders)
- product_id (产品ID)
- quantity (数量)
- unit_price (单价)
- discount_percent (折扣百分比)

## 业务逻辑

**活跃客户**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**收入计算**: 仅统计状态为 'completed' 的订单。使用 orders 表中的 total_amount 字段，该字段已计算折扣。

**客户生命周期价值 (CLV)**: 客户所有已完成订单金额的总和。

**高价值订单**: total_amount > 1000 的订单。

## 示例查询

-- 获取上季度收入排名前10的客户
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10;
"""
    },
    {   "name"          : "inventory_management",
        "description"   : "用于库存跟踪的数据库模式和业务逻辑，包括产品、仓库和库存水平。",
        "content"       : """# 库存管理模式

## 数据表

### products (产品表)
- product_id (主键)
- product_name (产品名称)
- sku (库存单位)
- category (类别)
- unit_cost (单位成本)
- reorder_point (再订货点：需要补货的最低库存水平)
- discontinued (是否停产)

### warehouses (仓库表)
- warehouse_id (主键)
- warehouse_name (仓库名称)
- location (位置)
- capacity (容量)

### inventory (库存表)
- inventory_id (主键)
- product_id (外键 -> products)
- warehouse_id (外键 -> warehouses)
- quantity_on_hand (现有库存量)
- last_updated (最后更新时间)

### stock_movements (库存移动记录表)
- movement_id (主键)
- product_id (外键 -> products)
- warehouse_id (外键 -> warehouses)
- movement_type (移动类型：inbound入库/outbound出库/transfer调拨/adjustment调整)
- quantity (数量：入库为正，出库为负)
- movement_date (移动日期)
- reference_number (参考编号)

## 业务逻辑

**可用库存**: 来自 inventory 表中 quantity_on_hand > 0 的记录

**需要补货的产品**: 所有仓库的 quantity_on_hand 总和小于或等于产品 reorder_point 的产品

**仅限活跃产品**: 排除 discontinued = true 的产品，除非专门分析已停产项目

**库存估值**: 每个产品的 quantity_on_hand * unit_cost

## 示例查询

-- 查找所有仓库中库存低于再订货点的产品
SELECT
    p.product_id,
    p.product_name,
    p.reorder_point,
    SUM(i.quantity_on_hand) as total_stock,
    p.unit_cost,
    (p.reorder_point - SUM(i.quantity_on_hand)) as units_to_reorder
FROM products p
JOIN inventory i ON p.product_id = i.product_id
WHERE p.discontinued = false
GROUP BY p.product_id, p.product_name, p.reorder_point, p.unit_cost
HAVING SUM(i.quantity_on_hand) <= p.reorder_point
ORDER BY units_to_reorder DESC;
"""
    }
]

In [3]:
# 2. 创建技能记载的工具
from langchain.tools import tool

@tool#(name = "load skill")
def load_skill(skill_name:str) -> str:
    """
    将技能的完整内容加载到代理的上下文中。

    当您需要关于如何处理特定类型请求的详细信息时，应使用此功能。这将为您提供该技能领域的全面说明、政策和指南。

    Args:
        skill_name: 要加载的技能名称（例如，“费用报销”，“旅行预订”）
    """

    for skill in SKILLS:
        if skill['name'] == skill_name:
            return f"已经加载的技能:{skill_name}\n\n{skill['content']}"
        
        
        #未找到技能
        available = ",".join(s['name'] for s in SKILLS)             #可以用的技能
        return f"未找到技能'{skill_name}',可用得技能有：{available}"

In [7]:
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage, AIMessage
from typing import Callable
from langchain.agents.middleware import ExtendedModelResponse

class SkillMiddleWare(AgentMiddleware):
    """将技能描述注入系统提示词的中间件"""

    tools = [load_skill]

    def __init__(self):
        skill_list = []
        for skill in SKILLS:
            skill_list.append(
                f"- **{skill['name']}**: {skill['description']}"
            )

        self.skill_prompt = "\n".join(skill_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse | AIMessage | ExtendedModelResponse:

        # 构造追加内容
        skill_addendum = (
            f"\n\n## 可用技能:\n{self.skill_prompt}\n\n"
            "当需要详细技能信息时，请调用 load_skill 工具。"
        )

        # ✅ 处理 system_message 为空的情况
        if request.system_message:
            content_blocks = list(request.system_message.content_blocks)
        else:
            content_blocks = []

        # 注入内容
        content_blocks.append({
            "type": "text",
            "text": skill_addendum
        })

        new_system_message = SystemMessage(content=content_blocks)

        # 构造新请求
        new_request = request.override(system_message=new_system_message)

        return handler(new_request)

In [17]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="ollama:qwen3.6:latest",
    base_url="http://192.168.8.21:11434"
    )
# 创建支持技能的代理
agent = create_agent(
    model,
    system_prompt=(
        "你是一位 SQL 查询助手，帮助用户针对业务数据库编写查询。"
    ),
    middleware=[SkillMiddleWare()],
    checkpointer=InMemorySaver(),     # 支持历史问答记忆
)

# 5. 测试渐进式披露的效果

In [18]:
from langchain_core.utils.uuid import uuid7         #随机生成32位或者64位uuid

# 此对话线程的配置。
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

# 请求提供一个 SQL 查询。
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "编写一个 SQL 查询，查找上个月所有订单金额超过 1000 美元的客户"
                ),
            }
        ]
    },
    config
)

# 打印对话内容。
for message in result["messages"]:
    if hasattr(message, 'pretty_print'):
        message.pretty_print()
    else:
        print(f"{message.type}: {message.content}")

================================ Human Message =================================

编写一个 SQL 查询，查找上个月所有订单金额超过 1000 美元的客户
================================== Ai Message ==================================
Tool Calls:
  load_skill (6bc49cff-0408-4bde-ba5b-c9ce68b76366)
 Call ID: 6bc49cff-0408-4bde-ba5b-c9ce68b76366
  Args:
    skill_name: sales_analytics
================================= Tool Message =================================
Name: load_skill

已经加载的技能:sales_analytics

# 销售分析模式

## 数据表

### customers (客户表)
- customer_id (主键)
- name (姓名)
- email (邮箱)
- signup_date (注册日期)
- status (状态: active活跃/inactive非活跃)
- customer_tier (客户等级: bronze铜/silver银/gold金/platinum铂金)

### orders (订单表)
- order_id (主键)
- customer_id (外键 -> customers)
- order_date (订单日期)
- status (状态: pending待处理/completed已完成/cancelled已取消/refunded已退款)
- total_amount (总金额)
- sales_region (销售区域: north北/south南/east东/west西)

### order_items (订单明细表)
- item_id (主键)
- order_id (外键 -> orders)
- product_id (产品ID)
- quantity (数量)
- unit_

In [16]:
from typing import TypedDict

class Skill(TypedDict):
    name: str
    description: str
    content: str

SKILLS = [
    {
        "name": "math_skill",
        "description": "解决数学计算问题",
        "content": "当遇到数学问题时，你可以一步一步推理并计算结果，当你使用这个技能时，你必须在回答前输出：『正在使用 math_skill』"
    }
]

In [17]:
from IPython.display import display, Markdown, HTML

def visualize_result(result):
    for msg in result['messages']:
        # 根据消息类型设置不同的背景颜色和标签
        if msg.__class__.__name__ == "HumanMessage":
            bg_color = "#e3f2fd"  # 淡蓝色
            role_label = "👤 用户"
        else:
            bg_color = "#f5f5f5"  # 浅灰色
            role_label = "🤖 AI 助手"
        
        # 构建 HTML 结构
        header = f"<div style='font-weight: bold; margin-bottom: 5px;'>{role_label}:</div>"
        # 使用 Markdown 渲染 content 里的数学公式和格式
        content_html = f"<div style='background-color: {bg_color}; padding: 15px; border-radius: 10px; margin-bottom: 20px; line-height: 1.6;'>"
        
        display(HTML(header))
        display(Markdown(msg.content))
        display(HTML("</div>"))

In [24]:
"""最简版 Skill 注入器"""
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import SystemMessage
from rich import print as rprint


@wrap_model_call
def skill_injector(request: ModelRequest, handler):
    
    # 1. 拿到当前 messages
    messages = request.messages
    rprint("===== BEFORE =====")
    rprint(messages)
    # 2. 拼接 skill
    skill_text = ""

    for skill in SKILLS:
        skill_text += f"""
技能名称: {skill['name']}
技能描述: {skill['description']}
技能内容: {skill['content']}
"""
    
    # 3. 注入 system prompt
    new_messages = [SystemMessage(content="你可以使用以下技能：\n" + skill_text)] + messages

    rprint("===== after =====")
    rprint(new_messages)

    request = request.override(messages=new_messages)
    
    # 4. 继续调用模型
    response = handler(request)
    
    return response

In [27]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from rich import print as rprint

model = init_chat_model(model="ollama:gemma4:e4b",base_url="http://192.168.8.21:11434")
agent = create_agent(model=model,middleware=[skill_injector])

result = agent.invoke({"messages": [{"role": "user", "content": "请使用你掌握的技能解释什么是斐波那契数列，并举例"}]})


# 直接打印 result 字典
rprint("*"*80)
rprint(result)
visualize_result(result)

===== BEFORE =====

[
    HumanMessage(
        content='请使用你掌握的技能解释什么是斐波那契数列，并举例',
        additional_kwargs={},
        response_metadata={},
        id='5286aa0d-46f7-42b4-a939-ae49a262f551'
    )
]

===== after =====

[
    SystemMessage(
        content='你可以使用以下技能：\n\n技能名称: math_skill\n技能描述: 解决数学计算问题\n技能内容: 
当遇到数学问题时，你可以一步一步推理并计算结果\n',
        additional_kwargs={},
        response_metadata={}
    ),
    HumanMessage(
        content='请使用你掌握的技能解释什么是斐波那契数列，并举例',
        additional_kwargs={},
        response_metadata={},
        id='5286aa0d-46f7-42b4-a939-ae49a262f551'
    )
]

********************************************************************************

{
    'messages': [
        HumanMessage(
            content='请使用你掌握的技能解释什么是斐波那契数列，并举例',
            additional_kwargs={},
            response_metadata={},
            id='5286aa0d-46f7-42b4-a939-ae49a262f551'
        ),
        AIMessage(
            content='斐波那契数列（Fibonacci 
Sequence）是数学中最著名、最迷人的数列之一。它最初是以意大利数学家列奥纳多·斐波那契（Leonardo 
Fibonacci）的名义广为人知的，但其规律在自然界中无处不在，具有高度的生物学和几何意义。\n\n虽然我掌握的技能是数学计算
，但我会结合我的知识体系，首先为您用文字解释其概念，然后利用我的推理能力进行具体的数列演示。\n\n---\n\n### 🌟 一、 
什么是斐波那契数列？\n\n斐波那契数列是一个特殊的整数数列，其特点是：**从第三项开始，每一项都是前两项的和。**\n\n**
核心规律：**\n\n如果我们将斐波那契数列记为 $F_n$，那么它遵循的数学递推关系是：\n$$F_n = F_{n-1} + 
F_{n-2}$$\n\n其中，$F_n$ 表示数列中的第 $n$ 个数，$F_{n-1}$ 是前一项，$F_{n-2}$ 
是前前一项。\n\n**起始点：**\n该数列的起始两项是固定的，通常约定为 $F_1 = 1$ 和 $F_2 = 1$（或者从 $F_0=0, F_1=1$ 
开始）。\n\n### 🌿 二、 斐波那契数列的数列展示（举例）\n\n如果使用常见的起始点 $1, 1$ 
开始，完整的斐波那契数列是：\n\n$$1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 
\\dots$$\n\n**【数学推理演示】**\n\n为了更清晰地展示其规律，我将使用我的数学技能来计算这个数列的几项，以证明每一项
都是前两项的和：\n\n**技能使用：math_skill**\n\n**目标：** 计算数列的前 7 项，并验证其规律。\n\n1.  **第 1 项 
(F₁):** 1\n2.  **第 2 项 (F₂):** 1\n3.  **第 3 项 (F₃):** $F_1 + F_2 = 1 + 1 = 2$\n4.  **第 4 项 (F₄):** $F_2 + F_3
= 1 + 2 = 3$\n5.  **第 5 项 (F₅):** $F_3 + F_4 = 2 + 3 = 5$\n6.  **第 6 项 (F₆):** $F_4 + F_5 = 3 + 5 = 8$\n7.  
**第 7 项 (F₇):** $F_5 + F_6 = 5 + 8 = 13$\n\n**结论：** 上述计算完美地展示了“每一项等于前两项之和”的规律。\n\n### 
🌍 三、 斐波那契数列的意义和应用\n\n斐波那契数列之所以如此著名，不仅是因为它的数学规律，更是因为它与**黄金比例 
($\\Phi$)** 之间的惊人联系，以及它在自然界和艺术设计中的广泛应用。\n\n**1. 与黄金比例 ($\\Phi$) 
的关系：**\n随着斐波那契数列的数字增大，相邻两项的比值（例如 
$\\frac{F_n}{F_{n-1}}$）会越来越接近一个神奇的常数，这个常数就是**黄金比例** $\\Phi$，其近似值约为 
**$1.618$**。黄金比例被认为是自然界中最和谐、最美观的比例。\n\n**2. 在自然界中的应用（黄金分割）：**\n*   
**植物的排列：** 
向日葵的种子排列、松果的螺旋结构，它们通常会以遵循斐波那契数字的角度进行排列，以实现最佳的采光和空间利用。\n*   
**生物的结构：** 某些扇贝壳、鹦鹉螺的螺旋生长也是遵循与黄金比例相关的曲线。\n*   **骨骼和葉片：** 
人体的骨骼结构和许多植物叶片叶序的排列规律也常与斐波那契数列相关。\n\n**3. 
在艺术与建筑中的应用：**\n从文艺复兴时期的建筑到现代的黄金分割摄影构图，该数列所蕴含的比例原则被认为是美学和和谐的
基础。',
            additional_kwargs={},
            response_metadata={
                'model': 'gemma4:e4b',
                'created_at': '2026-04-29T02:20:25.8799838Z',
                'done': True,
                'done_reason': 'stop',
                'total_duration': 336532611500,
                'load_duration': 307251340200,
                'prompt_eval_count': 71,
                'prompt_eval_duration': 291993100,
                'eval_count': 1718,
                'eval_duration': 27883375300,
                'logprobs': None,
                'model_name': 'gemma4:e4b',
                'model_provider': 'ollama'
            },
            id='lc_run--019dd704-9a75-76c1-a723-7d880df174d0-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 71, 'output_tokens': 1718, 'total_tokens': 1789}
        )
    ]
}

请使用你掌握的技能解释什么是斐波那契数列，并举例

斐波那契数列（Fibonacci Sequence）是数学中最著名、最迷人的数列之一。它最初是以意大利数学家列奥纳多·斐波那契（Leonardo Fibonacci）的名义广为人知的，但其规律在自然界中无处不在，具有高度的生物学和几何意义。

虽然我掌握的技能是数学计算，但我会结合我的知识体系，首先为您用文字解释其概念，然后利用我的推理能力进行具体的数列演示。

---

### 🌟 一、 什么是斐波那契数列？

斐波那契数列是一个特殊的整数数列，其特点是：**从第三项开始，每一项都是前两项的和。**

**核心规律：**

如果我们将斐波那契数列记为 $F_n$，那么它遵循的数学递推关系是：
$$F_n = F_{n-1} + F_{n-2}$$

其中，$F_n$ 表示数列中的第 $n$ 个数，$F_{n-1}$ 是前一项，$F_{n-2}$ 是前前一项。

**起始点：**
该数列的起始两项是固定的，通常约定为 $F_1 = 1$ 和 $F_2 = 1$（或者从 $F_0=0, F_1=1$ 开始）。

### 🌿 二、 斐波那契数列的数列展示（举例）

如果使用常见的起始点 $1, 1$ 开始，完整的斐波那契数列是：

$$1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, \dots$$

**【数学推理演示】**

为了更清晰地展示其规律，我将使用我的数学技能来计算这个数列的几项，以证明每一项都是前两项的和：

**技能使用：math_skill**

**目标：** 计算数列的前 7 项，并验证其规律。

1.  **第 1 项 (F₁):** 1
2.  **第 2 项 (F₂):** 1
3.  **第 3 项 (F₃):** $F_1 + F_2 = 1 + 1 = 2$
4.  **第 4 项 (F₄):** $F_2 + F_3 = 1 + 2 = 3$
5.  **第 5 项 (F₅):** $F_3 + F_4 = 2 + 3 = 5$
6.  **第 6 项 (F₆):** $F_4 + F_5 = 3 + 5 = 8$
7.  **第 7 项 (F₇):** $F_5 + F_6 = 5 + 8 = 13$

**结论：** 上述计算完美地展示了“每一项等于前两项之和”的规律。

### 🌍 三、 斐波那契数列的意义和应用

斐波那契数列之所以如此著名，不仅是因为它的数学规律，更是因为它与**黄金比例 ($\Phi$)** 之间的惊人联系，以及它在自然界和艺术设计中的广泛应用。

**1. 与黄金比例 ($\Phi$) 的关系：**
随着斐波那契数列的数字增大，相邻两项的比值（例如 $\frac{F_n}{F_{n-1}}$）会越来越接近一个神奇的常数，这个常数就是**黄金比例** $\Phi$，其近似值约为 **$1.618$**。黄金比例被认为是自然界中最和谐、最美观的比例。

**2. 在自然界中的应用（黄金分割）：**
*   **植物的排列：** 向日葵的种子排列、松果的螺旋结构，它们通常会以遵循斐波那契数字的角度进行排列，以实现最佳的采光和空间利用。
*   **生物的结构：** 某些扇贝壳、鹦鹉螺的螺旋生长也是遵循与黄金比例相关的曲线。
*   **骨骼和葉片：** 人体的骨骼结构和许多植物叶片叶序的排列规律也常与斐波那契数列相关。

**3. 在艺术与建筑中的应用：**
从文艺复兴时期的建筑到现代的黄金分割摄影构图，该数列所蕴含的比例原则被认为是美学和和谐的基础。